In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
# 모든 라이브러리를 설치 후에 세션을 다시 시작해주세요
# Colab 셀에 한 번만 실행
!pip install -q protobuf==3.20.3 gensim==4.3.1
!pip -q uninstall -y scipy
!pip -q install scipy==1.9.3
!pip -q install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 83.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.4/33.4 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 57.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages tha

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt

# 텍스트 데이터 전처리
import re
# import contractions
from collections import Counter
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize

# Tensorflow 라이브러리
import tensorflow as tf

# Tensorflow 전처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# word2vec 모델
from gensim.models import KeyedVectors
from gensim.models import Word2Vec

# Tensorflow 모델 구조
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import MultiHeadAttention, Conv1D, Bidirectional, LSTM, GlobalAveragePooling1D, Lambda, Attention, ReLU,Softmax
from tensorflow.keras.layers import Input, Embedding, Flatten, Layer, Multiply, Concatenate, Dropout, Dense, Reshape, Permute, Dot
from tensorflow.keras.utils import plot_model

# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils


In [ ]:
# 텍스트 패딩을 생성한 후, 리스트의 형태로 DataFrame에 저장
def create_pad_sequence(df, col, max_sequence_length = 10):

    # 토큰화 및 패딩
    text_sequence = tokenizer.texts_to_sequences(df[col])
    padding_result = pad_sequences(text_sequence, maxlen=max_sequence_length, padding='post')
    padding_result # 새로 만들어준 사전에서의 인덱스

    # 'padded_sequences' 열을 df에 추가하고, 패딩이 적용된 시퀀스의 내용을 할당
    df[col + '_padded_sequences'] = list(padding_result)

    # 'padded_sequences' 열의 데이터 타입을 object로 변경하여 각 요소를 리스트 형태로 저장
    df[col + '_padded_sequences'] = df[col + '_padded_sequences'].apply(lambda x: list(x))

# 데이터셋 로딩

In [ ]:
# 데이터 로딩
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.json')
print(df.shape)
df.head()

(66718, 15)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[-0.2068804801, -0.3263846636, 0.1318071485, -...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0685713813, 0.1392135769, -0.0395870171, -..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[0.062450625, -0.0707582533, 0.1557247639, -0....","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0461796485, 0.1261856854, -0.0177110955, -..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[-0.2690293193, -0.42407187820000003, 0.823390...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0621351376, 0.1333664954, -0.0194056146, -..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[0.1193233952, -0.5735993385, 0.40760195260000...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0678028613, 0.051634714000000005, -0.01779..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123198, -0.9643414617, 0.4619303048, -...","[-0.4373431206, -0.6204138398, 0.1469407529, -...","[-0.1105786338, 0.1126922667, -0.0412610471, -...","[-0.0794128701, 0.0517616905, -0.0255422704, -..."


# Text Embedding


In [ ]:
# Path to the Google News Word2Vec model
model_path = '/content/drive/MyDrive/GoogleNews-vectors-negative300.bin'
# Load the model
google_w2v_model = KeyedVectors.load_word2vec_format(model_path, binary=True)
# Print the shape of the model to confirm its size
print('Model shape:', google_w2v_model.vectors.shape)
# Output should be (3000000, 300) -> to do: (3000000개에서 사용되는 단어만 select)

Model shape: (3000000, 300)


In [ ]:
# 최대 사용할 단어 수와 시퀀스의 최대 길이
max_words = 100000

# 토크나이저 초기화 및 학습
tokenizer = Tokenizer(num_words=max_words)

In [ ]:
# lemmatized text 기준으로 사전 만들어주기
keyword = df['text'].explode().unique()
tokenizer.fit_on_texts(keyword)

In [ ]:
# 최종 단어 index의 길이가 max word를 넘는 상황, 65만개 dataset 기준
len(tokenizer.word_index)

47118

In [ ]:
# 단어 인덱스 맵 생성 및 임베딩 행렬 준비
# 임베딩 차원을 300으로 가정
word_index = tokenizer.word_index
total_words = min(max_words, len(word_index))
embedding_matrix = np.zeros((total_words, 300))

# 새로 만든 사전에 맞게 임베딩 해줘야함
# 임베딩 행렬의 크기 출력
for word, i in word_index.items():
    if i > max_words:
        continue
    if word in google_w2v_model:
        embedding_matrix[i] = google_w2v_model[word]

print(embedding_matrix)

[[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.08007812  0.10498047  0.04980469 ...  0.00366211  0.04760742
  -0.06884766]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]


In [ ]:
total_words

47118

In [ ]:
len(embedding_matrix)

47118

# User, Item Embedding을 위한 전처리
- user별, item별 임베딩을 생성한 다음, 전체 DataFrame에 concat할 예정
- 실행 시간 단축을 위한 방안

In [ ]:
df['user_id'].nunique()

7449

In [ ]:
df['asin'].nunique()

16099

In [ ]:
# user별, item별로 텍스트를 연결
user_text_df = df.groupby('user_id')['text'].apply(lambda x: ' '.join(x)).reset_index()
item_text_df = df.groupby('asin')['text'].apply(lambda x: ' '.join(x)).reset_index()

# column 변경
user_text_df.columns = ['user_id', 'user_text']
item_text_df.columns = ['asin', 'item_text']

# 결과 확인
print(user_text_df.shape)
print(item_text_df.shape)

user_text_df.head()

(7449, 2)
(16099, 2)


,user_id,user_text
0,AE22ZYH6WIY6WJHJRWOMPQCDJHHA,I ordered these as a gift for a loved one. It ...
1,AE23NZUELB4BYWLHKWJXAS73PTSQ,These packing cubes are very nice quality. The...
2,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,The vacuum sealed snowsuit came and while I wa...
3,AE24VPMWEEQD62YPOG53BW7JCGFA,This is a very nice set of heavy-duty workboot...
4,AE25JLQJ6LWKPHI3JFHBN5F4YN2A,"Unfortunately my dish arrived damaged, which i..."


# Create Embedding Matrix

In [ ]:
# sequence 길이 500으로 embedding 생성
create_pad_sequence(user_text_df,'user_text', max_sequence_length = 300)
create_pad_sequence(item_text_df,'item_text', max_sequence_length = 300)

# Data Concat

In [ ]:
# 원래의 데이터셋에 결합해서 최종 데이터셋을 만들어줌

df = pd.merge(df, user_text_df, on='user_id', how='left')
df = pd.merge(df, item_text_df, on='asin', how='left')

# User, Item label encoding
- 라벨 인코딩
- train, test 분리



In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

In [ ]:
# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

# train과 test data를 분리
train, test = train_test_split(df, shuffle=True, test_size=0.2, random_state=42)
train, valid = train_test_split(train, shuffle=True, test_size=0.125, random_state=42)

# index reset
train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
# inputs
user_text_train = np.array(train['user_text_padded_sequences'].tolist(),dtype='int64')
user_text_valid = np.array(valid['user_text_padded_sequences'].tolist(),dtype='int64')
user_text_test = np.array(test['user_text_padded_sequences'].tolist(),dtype='int64')

item_text_train = np.array(train['item_text_padded_sequences'].tolist(),dtype='int64')
item_text_valid = np.array(valid['item_text_padded_sequences'].tolist(),dtype='int64')
item_text_test = np.array(test['item_text_padded_sequences'].tolist(),dtype='int64')

# rating
y_train = train['rating'].to_numpy()
y_valid = valid['rating'].to_numpy()
y_test = test['rating'].to_numpy()

# AFRAM

In [ ]:
def attention_sliding_window_model(sequence_length, embedding_dim, window_size, num_aspects):

    # Input Layer
    inputs = Input(shape=(sequence_length,))  # Input shape: (sequence_length, embedding_dim)

    # Embedding Layer
    embedding_layer = Embedding(total_words, embedding_dim, weights=[embedding_matrix], trainable=False)(inputs)  # Shape: (None, sequence_length, embedding_dim)

    # Sliding window with CNN to capture encoded context (padding ensures correct sliding)
    conv_output = Conv1D(filters=embedding_dim, kernel_size=window_size, padding='same')(embedding_layer)

    conv_output = ReLU()(conv_output)  # Apply ReLU activation

    # Compute attention weights
    attention_weights = Dense(1, activation='relu')(conv_output)
    attention_weights = Softmax(axis=1)(attention_weights)

    # Apply attention to the CNN outputs (this is equivalent to applying the attention weights)
    weighted_word_vectors = Multiply()([embedding_layer, attention_weights])

    # # Extract word aspect feature representations
    word_aspect_features = Dense(num_aspects, use_bias=False)(weighted_word_vectors)

    # 가중합 후 평균
    attention_output = Attention(name="attention_layer")([word_aspect_features, word_aspect_features])

    weighted_sum = GlobalAveragePooling1D(name="global_average_pooling")(attention_output)

    # Model to return local aspect feature representations
    model = Model(inputs=inputs, outputs=weighted_sum)

    return model

In [ ]:
def bi_lstm_aspect_model(sequence_length, embedding_dim, lstm_units, num_aspects):

    # Input Layer
    inputs = Input(shape=(sequence_length,))  # Input shape: (sequence_length, embedding_dim)

    # Embedding Layer
    embedding_layer = Embedding(total_words, embedding_dim, weights=[embedding_matrix], trainable=False)(inputs)  # Shape: (None, sequence_length, embedding_dim)

    # Bi-LSTM operation
    bi_lstm_output = Bidirectional(LSTM(lstm_units, return_sequences=True))(embedding_layer)  # Shape: (None, sequence_length, 2*lstm_units)

    # Contextual aspect representation (p_g_{i,a})
    aspect_transform = Dense(num_aspects)
    contextual_aspect_representation = aspect_transform(bi_lstm_output)  # Shape: (None, sequence_length, num_aspects)

    # Use the same tensor as both the query and key in the Attention layer
    attention_output =  Attention(name="attention_layer")([contextual_aspect_representation, contextual_aspect_representation])  # Shape: (None, sequence_length, num_aspects)


    # Model to return global aspect feature representations
    model = Model(inputs=inputs, outputs=attention_output)

    return model

In [ ]:
def combined_model(sequence_length, embedding_dim, window_size, lstm_units, num_aspects, learning_rate, dropout_rate):

    user_inputs = Input(shape=(sequence_length,))
    item_inputs = Input(shape=(sequence_length,))

    # Get outputs from both models
    user_local_model = attention_sliding_window_model(sequence_length, embedding_dim, window_size, num_aspects)
    user_global_model = bi_lstm_aspect_model(sequence_length, embedding_dim, lstm_units, num_aspects)
    item_local_model = attention_sliding_window_model(sequence_length, embedding_dim, window_size, num_aspects)
    item_global_model = bi_lstm_aspect_model(sequence_length, embedding_dim, lstm_units, num_aspects)

    P_l_a_user = user_local_model(user_inputs)
    P_g_i_a_user = user_global_model(user_inputs)
    P_l_a_item = item_local_model(item_inputs)
    P_g_i_a_item = item_global_model(item_inputs)

    # Compute Pi,a = [(Pl_a - Pg_i_a) || (Pl_a + Pg_i_a)]
    user_diff = tf.keras.layers.Subtract()([P_l_a_user, P_g_i_a_user])
    user_add = tf.keras.layers.Add()([P_l_a_user, P_g_i_a_user])
    P_i_a_user = Multiply()([user_diff, user_add])

    item_diff = tf.keras.layers.Subtract()([P_l_a_item, P_g_i_a_item])
    item_add = tf.keras.layers.Add()([P_l_a_item, P_g_i_a_item])
    P_i_a_item = Multiply()([item_diff, item_add])

    user_attention_output = Attention(name="attention_layer1")([P_i_a_user, P_i_a_user])
    item_attention_output = Attention(name="attention_layer2")([P_i_a_item, P_i_a_item])

    user_attention_output=Flatten()(user_attention_output)
    item_attention_output=Flatten()(item_attention_output)

    user_rep = Dense(units=128)(user_attention_output)
    user_rep = Dropout(rate=0.5)(user_rep)
    user_rep = Dense(units=32)(user_rep)


    item_rep = Dense(units=128)(item_attention_output)
    item_rep = Dropout(rate=0.5)(item_rep)
    item_rep = Dense(units=32)(item_rep)

    rating_prediction = Dot(axes=1)([user_rep, item_rep])

    model = Model(inputs=[user_inputs,item_inputs], outputs=rating_prediction)
    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate), loss = "mse", metrics = ["mse", "mae"])

    return model

In [ ]:
# Example usage:
sequence_length = 300
embedding_dim = 300
window_size = 5
lstm_units = 5
num_aspects = 5
dropout_rate = 0.5
learning_rate = 0.0008 #0.0001

In [ ]:
model = combined_model(sequence_length, embedding_dim, window_size, lstm_units, num_aspects, learning_rate, dropout_rate)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 5)         │ 14,587,501 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 300, 5)    │ 14,147,695 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_2        │ (None, 5)         │ 14,587,501 │ input_layer_1[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_3        │ (None, 300, 5)    │ 14,147,695 │ input_layer_1[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract (Subtract) │ (None, 300, 5)    │          0 │ functional[0][0], │
│                     │                   │            │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 300, 5)    │          0 │ functional[0][0], │
│                     │                   │            │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_1          │ (None, 300, 5)    │          0 │ functional_2[0][… │
│ (Subtract)          │                   │            │ functional_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 300, 5)    │          0 │ functional_2[0][… │
│                     │                   │            │ functional_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 300, 5)    │          0 │ subtract[0][0],   │
│ (Multiply)          │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 300, 5)    │          0 │ subtract_1[0][0], │
│ (Multiply)          │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer1    │ (None, 300, 5)    │          0 │ multiply_2[0][0], │
│ (Attention)         │                   │            │ multiply_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer2    │ (None, 300, 5)    │          0 │ multiply_3[0][0], │
│ (Attention)         │                   │            │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1500)      │          0 │ attention_layer1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 1500)      │          0 │ attention_layer2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │    192,128 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128)       │    192,128 │ flatten_1[0][0] 

 Total params: 57,862,904 (220.73 MB)

 Trainable params: 1,321,304 (5.04 MB)

 Non-trainable params: 56,541,600 (215.69 MB)

In [ ]:
# Configuration of EarlyStopping
early_stopping = EarlyStopping(
    monitor ='val_loss',  # Monitor model's validation loss
    min_delta = 0.001,     # Minimum change to qualify as an improvement
    patience = 5,         # Stop after 5 epochs without improvement
    verbose = 1,           # Print messages
    mode = 'min',          # Stop training when the quantity monitored has stopped decreasing
    restore_best_weights = True  # Restore model weights from the epoch with the best value of the monitored quantity
)

In [ ]:
# 모델 학습
history = model.fit([user_text_train, item_text_train], y_train,
                    validation_data = ([user_text_valid, item_text_valid], y_valid),
                    epochs = 100,
                    batch_size = 64,
                    callbacks=[early_stopping]) #128 256 #논문-데이터 사이즈 64

Epoch 1/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 57s 48ms/step - loss: 3.7245 - mae: 1.4242 - mse: 3.7245 - val_loss: 0.5306 - val_mae: 0.5885 - val_mse: 0.5306
Epoch 2/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - loss: 0.7217 - mae: 0.6507 - mse: 0.7217 - val_loss: 0.4921 - val_mae: 0.5307 - val_mse: 0.4921
Epoch 3/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - loss: 0.6056 - mae: 0.5867 - mse: 0.6056 - val_loss: 0.5202 - val_mae: 0.4686 - val_mse: 0.5202
Epoch 4/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - loss: 0.5505 - mae: 0.5536 - mse: 0.5505 - val_loss: 0.5027 - val_mae: 0.4634 - val_mse: 0.5027
Epoch 5/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 34s 46ms/step - loss: 0.5103 - mae: 0.5316 - mse: 0.5103 - val_loss: 0.4784 - val_mae: 0.4906 - val_mse: 0.4784
Epoch 6/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - loss: 0.4943 - mae: 0.5167 - mse: 0.4943 - val_loss: 0.4836 - val_mae: 0.4614 - val_mse: 0.4836
Epoch 7/100
730/730 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - loss: 0.4809 - mae: 0.510

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_text_test, item_text_test])

# MAE
MAE = mean_absolute_error(y_test, predicted_ratings)
print(f"MAE: {MAE:.4f}")

# MSE
MSE = mean_squared_error(y_test, predicted_ratings)
print(f"MSE: {MSE:.4f}")

# RMSE
RMSE = np.sqrt(MSE)
print(f"RMSE: {RMSE:.4f}")

# MAPE
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100
print(f"MAPE: {MAPE:.4f}%")

417/417 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step
MAE: 0.4950
MSE: 0.4587
RMSE: 0.6773
MAPE: 14.3307%
